# 🚀 Vertex AI Deployment — FMCG Promotional Analytics
**Purpose:** Deploy UK and Southeast Asia XGBoost models to Vertex AI endpoints.  
**Run all cells top to bottom.** Deployment takes ~15 minutes per model.

| | Resource |
|---|---|
| Project | `YOUR_GCP_PROJECT_ID` |
| Region | `YOUR_REGION` |
| Bucket | `gs://YOUR_BUCKET_NAME` |
| UK Endpoint | `8363485774213021696` |
| Sea Endpoint | `1652277904500850688` |


## Step 1 — Install & Authenticate

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 1 — Install required libraries
# ══════════════════════════════════════════════════════════════════════
!pip install google-cloud-aiplatform google-cloud-storage xgboost==1.7.6 --quiet
print("✅ Libraries installed")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 2 — Load GCP credentials
# ══════════════════════════════════════════════════════════════════════
import os
import google.auth
import google.auth.transport.requests

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = (
    r"YOUR_ADC_PATH"
)

credentials, project = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
credentials.refresh(google.auth.transport.requests.Request())

print(f"✅ Credentials loaded")
print(f"   Project    : {project}")
print(f"   Token valid: {credentials.valid}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 3 — Imports & project config
# ══════════════════════════════════════════════════════════════════════
import os, pickle, shutil
import numpy as np
import pandas as pd
import xgboost as xgb
from google.cloud import aiplatform, storage

PROJECT_ID  = "YOUR_GCP_PROJECT_ID"
REGION      = "YOUR_REGION"
BUCKET_NAME = "YOUR_BUCKET_NAME"
BUCKET_URI  = f"gs://{BUCKET_NAME}"

# Sklearn container — compatible with XGBoost 1.7.6
SKLEARN_CONTAINER = "europe-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-0:latest"

# Initialise Vertex AI SDK
aiplatform.init(project=PROJECT_ID, location=REGION, credentials=credentials)

print(f"✅ SDK ready")
print(f"   XGBoost version : {xgb.__version__}  (must be 1.7.6)")
print(f"   Bucket          : {BUCKET_URI}")


## Step 2 — Prepare Model Artefacts

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 4 — Prepare staging folders and re-save models
# Models must be saved with XGBoost 1.7.6 for Vertex AI compatibility
# ══════════════════════════════════════════════════════════════════════
os.makedirs("staging/uk",   exist_ok=True)
os.makedirs("staging/sea", exist_ok=True)

# UK model — load from original pkl, re-save with 1.7.6
with open("models/xgb_uk_tuned.pkl", "rb") as f:
    uk_raw = pickle.load(f)

uk_reg = xgb.XGBRegressor()
uk_reg.load_model("models/xgb_uk_tuned.json")    # load from JSON (version-safe)
with open("staging/uk/model.pkl", "wb") as f:
    pickle.dump(uk_reg, f)
print("✅ UK model.pkl saved  (XGBoost 1.7.6)")

# Copy feature list
shutil.copy("models/feature_cols_uk.pkl", "staging/uk/feature_cols.pkl")
print("✅ UK feature_cols.pkl copied")

# Southeast Asia model
with open("models/xgb_sea_tuned.pkl", "rb") as f:
    id_raw = pickle.load(f)

id_reg = xgb.XGBRegressor()
id_reg.load_model("models/xgb_sea_tuned.json")
with open("staging/sea/model.pkl", "wb") as f:
    pickle.dump(id_reg, f)
print("✅ Southeast Asia model.pkl saved  (XGBoost 1.7.6)")

shutil.copy("models/feature_cols_sea.pkl", "staging/sea/feature_cols.pkl")
print("✅ Southeast Asia feature_cols.pkl copied")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 5 — Upload artefacts to GCS
# ══════════════════════════════════════════════════════════════════════
client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BUCKET_NAME)

for market, local_dir, gcs_prefix in [
    ("UK",        "staging/uk",   "artifacts/uk"),
    ("Southeast Asia", "staging/sea", "artifacts/sea"),
]:
    for fname in os.listdir(local_dir):
        local_path = os.path.join(local_dir, fname)
        blob_path  = f"{gcs_prefix}/{fname}"
        bucket.blob(blob_path).upload_from_filename(local_path)
        print(f"  ✅ {market}: gs://{BUCKET_NAME}/{blob_path}")

print("\n✅ All artefacts uploaded to GCS")


## Step 3 — Register Models in Vertex AI

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 6 — Register UK model in Vertex AI Model Registry
# ══════════════════════════════════════════════════════════════════════
uk_model = aiplatform.Model.upload(
    display_name                = "uk-xgb-promo-model-v4",
    artifact_uri                = f"{BUCKET_URI}/artifacts/uk",
    serving_container_image_uri = SKLEARN_CONTAINER,
    description                 = "XGBoost 1.7.6 promo forecaster — UK market, R²=0.81, 97 features",
    labels                      = {"market": "uk", "version": "v4"},
)
print(f"✅ UK model registered")
print(f"   Resource: {uk_model.resource_name}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 7 — Register Southeast Asia model in Vertex AI Model Registry
# ══════════════════════════════════════════════════════════════════════
sea_model = aiplatform.Model.upload(
    display_name                = "sea-xgb-promo-model-v4",
    artifact_uri                = f"{BUCKET_URI}/artifacts/sea",
    serving_container_image_uri = SKLEARN_CONTAINER,
    description                 = "XGBoost 1.7.6 promo forecaster — Southeast Asia, R²=0.70, 78 features",
    labels                      = {"market": "southeast_asia", "version": "v4"},
)
print(f"✅ Southeast Asia model registered")
print(f"   Resource: {sea_model.resource_name}")


## Step 4 — Deploy to Endpoints

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 8 — Load existing endpoints
# (These were created on Saturday — no need to recreate)
# ══════════════════════════════════════════════════════════════════════
uk_endpoint = aiplatform.Endpoint(
    "projects/128825737789/locations/YOUR_REGION/endpoints/8363485774213021696"
)
sea_endpoint = aiplatform.Endpoint(
    "projects/128825737789/locations/YOUR_REGION/endpoints/1652277904500850688"
)

print("✅ Endpoints loaded")
print(f"   UK   : {uk_endpoint.resource_name}")
print(f"   Sea : {sea_endpoint.resource_name}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 9 — Deploy UK model to endpoint
# ⏳ Takes 10-15 minutes — do not interrupt
# ══════════════════════════════════════════════════════════════════════
print("Deploying UK model... please wait (10–15 min)")

uk_endpoint.deploy(
    model                       = uk_model,
    deployed_model_display_name = "uk-xgb-v4",
    machine_type                = "n1-standard-2",
    min_replica_count           = 1,
    max_replica_count           = 2,
    traffic_split               = {"0": 100},
    sync                        = False,    # non-blocking — check GCP console
)

print("✅ UK deployment started — check GCP Console → Vertex AI → Endpoints")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 10 — Deploy Southeast Asia model to endpoint
# ⏳ Takes 10-15 minutes — do not interrupt
# ══════════════════════════════════════════════════════════════════════
print("Deploying Southeast Asia model... please wait (10–15 min)")

sea_endpoint.deploy(
    model                       = sea_model,
    deployed_model_display_name = "sea-xgb-v4",
    machine_type                = "n1-standard-2",
    min_replica_count           = 1,
    max_replica_count           = 2,
    traffic_split               = {"0": 100},
    sync                        = False,
)

print("✅ Southeast Asia deployment started — check GCP Console → Vertex AI → Endpoints")


## Step 5 — Test Live Endpoints

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 11 — Test UK endpoint prediction
# Only run this AFTER GCP Console shows the endpoint as Active ✅
# ══════════════════════════════════════════════════════════════════════
import pickle
import numpy as np

# Build a test instance using real median feature values
feat_cols = pickle.load(open("models/feature_cols_uk.pkl", "rb"))

# Use the median row we saved earlier
import json
medians = json.load(open("models/feature_medians_uk.json"))
test_instance = [float(medians.get(col, 0.0)) for col in feat_cols]

# Call the live endpoint
response = uk_endpoint.predict(instances=[test_instance])
log_pred = response.predictions[0]
pred_vol = float(np.expm1(log_pred))

print(f"✅ UK endpoint live prediction: {pred_vol:,.0f} units")
if pred_vol > 0:
    print("✅ Non-zero prediction — endpoint is working correctly!")
else:
    print("⚠️  Zero prediction — check endpoint deployment status")


## Step 6 — Cost Control

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 12 — Undeploy models when not in use (stops billing)
# Run this at the end of each session
# To redeploy: run Cells 8-10 again
# ══════════════════════════════════════════════════════════════════════

# Undeploy UK
for m in uk_endpoint.list_models():
    print(f"Undeploying UK: {m.display_name}...")
    uk_endpoint.undeploy(deployed_model_id=m.id)
    print("  ✅ Done")

# Undeploy Southeast Asia
for m in sea_endpoint.list_models():
    print(f"Undeploying Sea: {m.display_name}...")
    sea_endpoint.undeploy(deployed_model_id=m.id)
    print("  ✅ Done")

print("\n✅ All models undeployed — billing stopped")
print("   Endpoints still exist and can be redeployed anytime")
